## Combination

In [174]:
print(list(df.columns.to_list()))

['game_id', 'action_number', 'period', 'game_clock_secs', 'team_tricode', 'player_id', 'player_name', 'foul_type', 'description']


In [216]:
import duckdb
import pandas as pd

# Path to your file
file_path = '../data_w/feature_store/player_ratings.parquet'

# Load the data using DuckDB
# We'll grab the first 5 rows for a better look
df = duckdb.query(f"SELECT * FROM '{file_path}' LIMIT 5").df()

# View the interactive table
df.head()

,player_id,player_name,as_of_game_id,adjusted_plus_minus,games_fitted
0,73,,0022500002,0.0,0
1,149,,0022500002,0.0,0
2,677,,0022500002,0.0,0
3,683,,0022500002,0.0,0
4,698,,0022500002,0.0,0


In [176]:
print(list(df.columns.to_list()))

['game_id', 'action_number', 'period', 'game_clock_secs', 'team_tricode', 'player_id', 'player_name', 'foul_type', 'description']


In [212]:
import duckdb

# Path setup
p_path = '../data_w/raw/possessions_202526.parquet'
f_path = '../data_w/raw/foul_events_202526.parquet'
s_path = '../data_w/raw/substitution_events_202526.parquet'
t_path = '../data_w/raw/timeout_events_202526.parquet'
g_path = '../data_w/raw/games_202526.parquet' # NEW

query = f"""
WITH events AS (
    -- Possessions 
    SELECT 
        game_id, period, game_clock_secs, 1 AS event_priority, 'POSSESSION' AS event_type,
        possession_id, COALESCE(points, 0) AS points_scored,
        period_wall_clock, home_lineup_id, away_lineup_id, home_score, away_score, possessing_team,
        play_type, player_id, shot_value, shot_distance, shot_x, shot_y,
        NULL AS foul_type,
        NULL AS timeout_type,
        NULL AS player_in_id,
        NULL AS player_out_id,
        possessing_team as active_team
    FROM '{p_path}'

    UNION ALL 
    
    -- Fouls
    SELECT 
        game_id, period, game_clock_secs, 2 AS event_priority, 'FOUL' AS event_type,
        NULL AS possession_id, 0 AS points_scored,
        NULL AS period_wall_clock, NULL AS home_lineup_id, NULL AS away_lineup_id, NULL AS home_score, NULL AS away_score, NULL AS possessing_team,
        NULL AS play_type, player_id, NULL AS shot_value, NULL AS shot_distance, NULL AS shot_x, NULL AS shot_y,
        foul_type,
        NULL AS timeout_type,
        NULL AS player_in_id,
        NULL AS player_out_id,
        team_tricode as active_team
    FROM '{f_path}'

    UNION ALL 
    
    -- Timeouts
    SELECT 
        game_id, period, game_clock_secs, 3 AS event_priority, 'TIMEOUT' AS event_type,
        NULL AS possession_id, 0 AS points_scored,
        NULL AS period_wall_clock, NULL AS home_lineup_id, NULL AS away_lineup_id, NULL AS home_score, NULL AS away_score, NULL AS possessing_team,
        NULL AS play_type, NULL AS player_id, NULL AS shot_value, NULL AS shot_distance, NULL AS shot_x, NULL AS shot_y,
        NULL AS foul_type,
        timeout_type,
        NULL AS player_in_id,
        NULL AS player_out_id,
        team_tricode as active_team
    FROM '{t_path}'

    UNION ALL 
    
    -- Substitutions
    SELECT 
        game_id, period, game_clock_secs, 4 AS event_priority, 'SUBSTITUTION' AS event_type,
        NULL AS possession_id, 0 AS points_scored,
        NULL AS period_wall_clock,
        home_lineup_id, 
        away_lineup_id, 
        NULL AS home_score, NULL AS away_score, NULL AS possessing_team,
        NULL AS play_type, NULL AS player_id, NULL AS shot_value, NULL AS shot_distance, NULL AS shot_x, NULL AS shot_y,
        NULL AS foul_type,
        NULL AS timeout_type,
        player_in_id,
        player_out_id,
        team_tricode as active_team
    FROM '{s_path}'
),
state_tracker AS (
    SELECT 
        *,
        SUM(CASE WHEN event_type = 'TIMEOUT' THEN 1 ELSE 0 END) OVER current_game_window AS cumulative_timeouts,
        LAST_VALUE(period_wall_clock IGNORE NULLS) OVER current_game_window AS current_period_wall_clock,
        LAST_VALUE(possession_id IGNORE NULLS) OVER current_game_window AS current_possession_id,
        LAST_VALUE(home_lineup_id IGNORE NULLS) OVER current_game_window AS current_home_lineup_id,
        LAST_VALUE(away_lineup_id IGNORE NULLS) OVER current_game_window AS current_away_lineup_id,
        LAST_VALUE(home_score IGNORE NULLS) OVER current_game_window AS current_home_score,
        LAST_VALUE(away_score IGNORE NULLS) OVER current_game_window AS current_away_score,
        LAST_VALUE(possessing_team IGNORE NULLS) OVER current_game_window AS current_possessing_team,
        SUM(points_scored) OVER (
            PARTITION BY game_id, period
            ORDER BY (720 - game_clock_secs) ASC
            RANGE BETWEEN 180 PRECEDING AND CURRENT ROW
        ) AS points_scored_last_3m
    FROM events
    WINDOW current_game_window AS (
        PARTITION BY game_id
        ORDER BY period ASC, game_clock_secs DESC, event_priority ASC
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    )
)
-- STEP 3: Output Feed with Games Joined
SELECT 
    st.game_id, 
    g.home_team,  -- NEW
    g.away_team,  -- NEW
    CASE 
        WHEN st.active_team = 'home' THEN g.home_team
        WHEN st.active_team = 'away' THEN g.away_team
        ELSE st.active_team 
    END AS active_team,
    -- st.active_team,
    st.period, 
    st.game_clock_secs,
    st.current_period_wall_clock AS period_wall_clock, 
    st.event_type, 
    COALESCE(st.play_type, st.event_type) AS play_type,
    
    st.current_possession_id AS possession_id,
    st.current_home_lineup_id AS home_lineup_id,
    st.current_away_lineup_id AS away_lineup_id,
    st.current_home_score AS home_score,
    st.current_away_score AS away_score,
    st.current_possessing_team AS possessing_team,
    
    st.foul_type,
    st.points_scored,
    st.player_id,
    st.shot_value,
    st.shot_distance,
    st.shot_x,
    st.shot_y,
    
    st.player_in_id,
    st.player_out_id,
    
    st.cumulative_timeouts,
    st.points_scored_last_3m

FROM state_tracker st
LEFT JOIN '{g_path}' g 
  ON st.game_id = g.game_id 
ORDER BY st.game_id, st.period ASC, st.game_clock_secs DESC, st.event_priority ASC;
"""

df_mega_1 = duckdb.query(query).df()
df_mega_1.head(30)


,game_id,home_team,away_team,active_team,period,game_clock_secs,period_wall_clock,event_type,play_type,possession_id,...,points_scored,player_id,shot_value,shot_distance,shot_x,shot_y,player_in_id,player_out_id,cumulative_timeouts,points_scored_last_3m
0,0022400001,NaN,NaN,NaN,1,697.0,7:11 PM EST,POSSESSION,Stop,1,...,0,0,0,0,0,0,<NA>,<NA>,0.0,0.0
1,0022400001,NaN,NaN,NaN,1,682.0,7:11 PM EST,POSSESSION,Stop,2,...,0,0,0,0,0,0,<NA>,<NA>,0.0,0.0
2,0022400001,NaN,NaN,NaN,1,677.0,7:11 PM EST,POSSESSION,Turnover,3,...,0,1630552,0,0,0,0,<NA>,<NA>,0.0,0.0
3,0022400001,NaN,NaN,NaN,1,657.0,7:11 PM EST,POSSESSION,Turnover,4,...,0,1627759,0,0,0,0,<NA>,<NA>,0.0,0.0
4,0022400001,NaN,NaN,NaN,1,655.0,7:11 PM EST,POSSESSION,Turnover,5,...,0,1630700,0,0,0,0,<NA>,<NA>,0.0,0.0
5,0022400001,NaN,NaN,NaN,1,654.0,7:11 PM EST,POSSESSION,Turnover,6,...,0,1628401,0,0,0,0,<NA>,<NA>,0.0,0.0
6,0022400001,NaN,NaN,NaN,1,650.0,7:11 PM EST,POSSESSION,Made Shot,7,...,3,1630552,3,26,157,203,<NA>,<NA>,0.0,3.0
7,0022400001,NaN,NaN,NaN,1,635.0,7:11 PM EST,POSSESSION,Made Shot,8,...,3,1627759,3,27,102,253,<NA>,<NA>,0.0,6.0
8,0022400001,NaN,NaN,NaN,1,624.0,7:11 PM EST,POSSESSION,Free Throw,9,...,0,203991,0,0,0,0,<NA>,<NA>,0.0,6.0
9,0022400001,NaN,NaN,BOS,1,624.0,7:11 PM EST,FOUL,FOUL,9,...,0,1627759,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0.0,6.0


In [204]:
# 1. Check the exact percentage of Nulls
null_pct = df_mega_1[['home_team', 'away_team']].isnull().mean() * 100
print("--- Null Percentage ---")
print(null_pct)

--- Null Percentage ---
home_team    47.814304
away_team    47.814304
dtype: float64
